## Ricavo funzione
Fit dei dati della misura con solo sorgente + primo polarizzatore.
Se la sorgente è parzialmente polarizzata: I1(θ) = (I0/2)[1 + p cos(2θ − 2φ0)], cioè periodo 180° (2θ), non 4θ.

In [22]:
import ROOT
import numpy as np
import pandas as pd

ROOT.gStyle.SetOptFit(1111)      # box con chi2/ndf, probabilità e parametri (metti 0 per toglierlo)
ROOT.gStyle.SetFitFormat("4.2e")

### 1. Dati

In [23]:
FILE = "DatiSorgenteSim.xlsx"
FOGLIO = "Polarizzata"
N_SCARTA = 0                     # quanti punti iniziali scartare (0 = nessuno)

df = pd.read_excel(FILE, sheet_name=FOGLIO)
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.iloc[N_SCARTA:]

x     = df["θ [rad]"].to_numpy(dtype=float)
y     = df["V"].to_numpy(dtype=float)
err_x = df["sigma_θ"].to_numpy(dtype=float)
err_y = df["sigma_V"].to_numpy(dtype=float)

### 2. Fit

In [24]:
# Modello: V(θ) = A * (1 + E*cos(2θ - F)) * (1 + D*θ)
#   A = livello medio,  E = grado di polarizzazione P,  F = 2*φ0 (fase),  D = deriva lineare
modello = "[0]*(1 + [1]*cos(2*x - [2]))*(1 + [3]*x)"

fit_f = ROOT.TF1("fit_f", modello, x.min(), x.max())
E0 = (y.max() - y.min())/(y.max() + y.min())                  # valore di partenza di E
fit_f.SetParameters(y.mean(), E0, 2*x[np.argmax(y)], 0)       # F parte da 2*(posizione del massimo)
fit_f.SetParNames("A", "E", "F", "D")

grafico = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)    
grafico.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
grafico.SetMarkerStyle(20)
grafico.SetMarkerSize(0.7)
grafico.SetMarkerColor(ROOT.kAzure + 2)
grafico.SetLineColor(ROOT.kAzure + 3)

fit_res = grafico.Fit(fit_f, "S")

print(f"Fit status: {fit_res.Status()}")
for i, nome in enumerate(["A", "E", "F", "D"]):
    print(f"{nome} = {fit_f.GetParameter(i)} ± {fit_f.GetParError(i)}")
print(f"Chi2 = {fit_res.Chi2()}, ndf = {fit_res.Ndf()}")

# (E, F) e (-E, F+π) descrivono la stessa curva scelgo E positivo e F tra 0 e 2π
E_new, errE_new = fit_f.GetParameter(1), fit_f.GetParError(1)
F_new, errF_new = fit_f.GetParameter(2), fit_f.GetParError(2)
if E_new < 0: # qui controllo che E sia positivo altrimenti visto che il coseno è pari, cambio segno a E e aggiungo π a F
    E_new, F_new = -E_new, F_new + np.pi
F_new = F_new % (2*np.pi)

print(f"P = E = {E_new*100:.2f} ± {errE_new*100:.2f} %")
print(f"F = {F_new:.3f} ± {errF_new:.3f} rad = {np.degrees(F_new):.1f}°")

Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      142.752
NDf                       =          196
Edm                       =  7.60254e-06
NCalls                    =           94
A                         =      3.50321   +/-   0.00173536  
E                         =    0.0597946   +/-   0.000353358 
F                         =      2.72184   +/-   0.00578827  
D                         =  -0.00129809   +/-   0.00012435  

A = 3.5032111759004736 ± 0.0017353622186882798
E = 0.05979460888443228 ± 0.00035335804962457524
F = 2.7218385030525503 ± 0.0057882692407990955
D = -0.0012980886688174714 ± 0.00012434958072910944
Chi2 = 142.75173348545516, ndf = 196
P = E = 5.98 ± 0.04 %
F = 2.722 ± 0.006 rad = 155.9°


### 3. Grafico

In [25]:
c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()
grafico.Draw("AP")

leg = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg.AddEntry(grafico, "Dati", "pe")
leg.AddEntry(fit_f, "Fit", "l")
leg.Draw()

c1.Draw()
c1.SaveAs("Immagini/sorgente_fit.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/sorgente_fit.png has been created


## Confronto
Confronto E ed F ricavati ora (solo sorgente + P1) con quelli ricavati dalla misura completa di Malus, dove la sorgente veniva descritta dal fattore (1 + E cos(2θ - F)). Se la causa è la polarizzazione parziale della sorgente i due risultati devono essere compatibili.

Il confronto di F ha senso solo se la scala del polarizzatore (lo zero) e la posizione/orientazione della lampada non sono cambiate rispetto alla misura di prima.

In [26]:
# valori ottenuti dal fit della misura di Malus
E_old, errE_old = 0.0673, 0.0024
F_old, errF_old = 2.777, 0.036            # rad

#  confronto di E 
diff_E = E_new - E_old                            # di quanto differiscono
err_diff_E = np.sqrt(errE_new**2 + errE_old**2)       # incertezza della differenza
sigma_E = diff_E / err_diff_E                            # differenza in unità di sigma

#  confronto di F  
diff_F = F_new - F_old

# riporto la differenza tra -π e +π
if diff_F > np.pi:
    diff_F = diff_F - 2*np.pi
if diff_F < -np.pi:
    diff_F = diff_F + 2*np.pi

err_diff_F = np.sqrt(errF_new**2 + errF_old**2)
sigma_F = diff_F / err_diff_F

print(f"E: ora = {E_new*100:.2f} ± {errE_new*100:.2f} %   prima = {E_old*100:.2f} ± {errE_old*100:.2f} %   differenza = {sigma_E:.1f} sigma")
print(f"F: ora = {F_new:.3f} ± {errF_new:.3f} rad   prima = {F_old:.3f} ± {errF_old:.3f} rad   differenza = {sigma_F:.1f} sigma")

if abs(sigma_E) < 2 and abs(sigma_F) < 2:
    print("Compatibili entro 2 sigma: la polarizzazione della sorgente spiega l'onda trovata.")
else:
    print("NON compatibili: la polarizzazione della sorgente non basta a spiegare l'onda.")

E: ora = 5.98 ± 0.04 %   prima = 6.73 ± 0.24 %   differenza = -3.1 sigma
F: ora = 2.722 ± 0.006 rad   prima = 2.777 ± 0.036 rad   differenza = -1.5 sigma
NON compatibili: la polarizzazione della sorgente non basta a spiegare l'onda.
